# 레슨 07 — 안정적인 selector와 wait

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/07/%EB%A0%88%EC%8A%A8%2007%20%E2%80%94%20%EC%95%88%EC%A0%95%EC%A0%81%EC%9D%B8%20selector%EC%99%80%20wait.ipynb)

이 노트북은 읽기와 따라하기용 강의 노트북이다. 학생은 셀을 위에서 아래로 실행하며 웹 자동화에서 상태가 어떻게 유지되고 화면 조작이 어떤 순서로 기록되는지 확인한다. 안정적인 selector와 wait는 실제 사이트 대신 합성 fixture로 안전하게 연습한다.

## 학습 목표

1. 불안정한 class selector와 안정적인 data-testid selector를 구분한다.
2. 요소가 늦게 나타나는 상황을 wait로 처리한다.
3. timeout 실패를 명확히 기록한다.
4. wait 케이스를 CSV 기반으로 반복 검증한다.
5. selector 추천표와 결과 로그를 저장한다.

---

## 1. 안정 selector 기준

자동화에서는 화면 문구와 임의 class보다 data-testid, role, 고유 id가 안정적이다.


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/07/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

class MiniLocator:
    def __init__(self, page, selector):
        self.page = page
        self.selector = selector
    def elements(self):
        return self.page.soup.select(self.selector)
    def count(self):
        return len(self.elements())
    def first(self):
        items = self.elements()
        if not items:
            raise ValueError(f'no element for {self.selector}')
        return items[0]
    def text_content(self):
        return self.first().get_text(' ', strip=True)
    def all_text_contents(self):
        return [el.get_text(' ', strip=True) for el in self.elements()]
    def get_attribute(self, name):
        return self.first().get(name)
    def fill(self, value):
        self.first()['value'] = str(value)
    def click(self):
        return self.page._click(self.first())

class MiniPage:
    def __init__(self, html):
        self.soup = BeautifulSoup(html, 'html.parser')
        self.step = 0
        self.log = []
    def locator(self, selector):
        return MiniLocator(self, selector)
    def text_content(self, selector):
        return self.locator(selector).text_content()
    def fill(self, selector, value):
        self.locator(selector).fill(value)
        self.log.append({'action': 'fill', 'selector': selector, 'value': str(value)})
    def click(self, selector):
        result = self.locator(selector).click()
        self.log.append({'action': 'click', 'selector': selector, 'result': result})
        return result
    def _value(self, selector):
        el = self.soup.select_one(selector)
        return '' if el is None else el.get('value', '')
    def _click(self, el):
        action = el.get('data-action', '')
        if action == 'submit-profile':
            name = self._value('#student-name')
            course = self._value('#course-name')
            memo = self._value('#memo')
            out = self.soup.select_one('#result')
            out.string = f'{name} / {course} / {memo}'
            out['data-state'] = 'submitted'
            return 'submitted'
        if action == 'toggle-complete':
            target = self.soup.select_one(el.get('data-target', ''))
            if target:
                target['data-status'] = 'done' if target.get('data-status') != 'done' else 'pending'
                return target['data-status']
        if action == 'open-tab':
            target_id = el.get('data-target')
            for panel in self.soup.select('[role="tabpanel"]'):
                panel['hidden'] = 'true'
            target = self.soup.select_one(f'#{target_id}')
            if target and target.has_attr('hidden'):
                del target['hidden']
            return target_id
        return action or 'clicked'
    def visible_elements(self, selector):
        items = []
        for el in self.soup.select(selector):
            delay = int(el.get('data-delay-step', '0'))
            hidden = el.has_attr('hidden') or el.get('aria-hidden') == 'true'
            if delay <= self.step and not hidden:
                items.append(el)
        return items
    def tick(self):
        self.step += 1
        return self.step
    def wait_for_selector(self, selector, timeout_steps=5):
        for _ in range(timeout_steps + 1):
            items = self.visible_elements(selector)
            if items:
                return items[0]
            self.tick()
        raise TimeoutError(f'timeout waiting for {selector}')

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

lab = MiniPage(load_text('selector_lab.html'))
print(lab.locator('[data-testid="save-button"]').text_content())


---

## 2. wait가 필요한 이유

실제 화면은 네트워크와 렌더링 때문에 늦게 나타나는 요소가 있다. 조건이 맞을 때까지 기다리는 방식이 sleep보다 안정적이다.


In [ ]:
page = MiniPage(load_text('dynamic_dashboard.html'))
el = page.wait_for_selector('[data-testid="metric-submit"]', timeout_steps=2)
print(page.step, el.get_text(' ', strip=True))


---

## 3. 실패도 기록하기

없는 요소를 기다릴 때는 TimeoutError가 나야 잘못된 selector를 빨리 수정할 수 있다.


In [ ]:
try:
    page.wait_for_selector('[data-testid="missing"]', timeout_steps=1)
except Exception as error:
    print(type(error).__name__)


---

## 데이터 출처와 안전 규칙

이 레슨의 파일은 모두 수업용 합성 데이터다. 실제 사이트의 개인정보, 로그인 정보, 유료 콘텐츠를 포함하지 않는다. 실제 사이트로 확장할 때는 robots.txt, 이용 약관, 요청 간격, 개인정보 여부를 먼저 확인한다. 수업 중에는 fixture를 반복 실행하며 구조를 익히고, 외부 사이트를 빠르게 반복 요청하지 않는다.

---

## 수업 운영 메모

이 절은 수업 중 교사가 질문으로 풀어낼 수 있는 운영형 설명이다. 학생이 셀을 실행한 뒤 결과만 맞히지 않고 자동화 절차를 말로 설명하도록 돕는다.

### 1. 안정 selector의 우선순위

자동화 selector는 예쁘게 보이는 class보다 의미가 고정된 data-testid, role, label, id를 우선한다. 학생에게 class가 짧고 쉬워 보여도 배포 때 바뀔 수 있다는 점을 실제 사례로 설명한다.

### 2. wait와 sleep의 차이

sleep은 시간을 기다리고 wait는 조건을 기다린다. 화면이 빨리 뜨면 sleep은 시간을 낭비하고, 늦게 뜨면 실패할 수 있으므로 조건 기반 wait가 유지보수에 유리하다.

### 3. timeout을 숨기지 않기

요소가 끝까지 나타나지 않으면 명확한 TimeoutError가 나야 한다. 실패를 빈 문자열로 바꾸면 이후 단계에서 원인을 잃어버리므로 학생 답안에는 실패 메시지가 남아야 한다.

### 4. 동적 DOM 읽기

동적 화면은 처음 HTML에 모든 값이 있는 것이 아니다. 이 레슨의 fixture는 step이 진행되며 보이는 요소가 달라지도록 만들어, 학생이 기다림의 필요성을 눈으로 확인하게 한다.

### 5. selector 추천표

운영 코드에는 어떤 selector를 선택했는지 이유를 남기면 좋다. data-testid를 사용한 이유, 대체 selector, 피해야 할 selector를 표로 정리하면 다음 수정자가 빠르게 이해한다.

### 6. 테스트 케이스 반복

wait_cases.csv처럼 조건을 표로 만들면 여러 selector와 timeout 값을 반복 검증할 수 있다. 자동화 스크립트가 커질수록 이런 표 기반 점검이 수동 클릭보다 빠르다.

### 7. 수업 피드백

학생이 실패하면 timeout을 무작정 늘리기보다 selector가 맞는지, hidden 상태인지, step이 증가했는지 순서대로 확인하게 한다. 이 순서가 실제 브라우저 자동화 디버깅과 같다.

## selector 우선순위 표

| 우선순위 | selector 기준 | 권장 이유 | 주의점 |
|---:|---|---|---|
| 1 | `data-testid` | 테스트와 자동화를 위해 의도적으로 둔 속성 | 실제 서비스에 없으면 개발자와 협의 필요 |
| 2 | `role`, `aria-label` | 접근성 구조와 연결되어 의미가 안정적 | role이 중복될 수 있어 범위 제한 필요 |
| 3 | 고유 `id` | 짧고 빠르게 선택 가능 | 디자인용 id인지 기능용 id인지 확인 필요 |
| 4 | `name`, `data-target` | 폼과 연결 대상이 명확함 | 값이 동적으로 바뀌는지 확인 필요 |
| 5 | class | 스타일 변경에 취약함 | 유틸리티 class 조합은 피함 |
| 6 | 화면 문구 전체 | 번역, 띄어쓰기, 기획 문구 변경에 취약함 | 보조 검증에는 가능하지만 핵심 selector로는 부적합 |

학생이 selector를 고를 때는 “짧은가”보다 “왜 오래 유지될 수 있는가”를 말하게 한다. `selector_lab.html`의 `.btn.primary.x82`는 지금은 동작하지만 배포 때 class hash가 바뀌면 깨질 수 있다. 반면 `[data-testid="save-button"]`은 테스트용 계약이므로 유지될 가능성이 높다. 이 차이를 눈으로 확인시키는 것이 7강의 핵심이다.

## wait 정책을 세우는 기준

wait는 긴 timeout을 주는 기술이 아니다. 어떤 조건이 만족되면 다음 단계로 넘어갈지 정하는 정책이다. `metric-active`처럼 바로 보이는 값은 timeout 0으로도 읽힌다. `metric-submit`, `metric-pass`처럼 늦게 보이는 값은 필요한 step만큼 기다려야 한다. 반대로 존재하지 않는 selector는 끝까지 기다린 뒤 명확하게 실패해야 한다.

실제 Playwright에서는 `await expect(locator).toBeVisible()`처럼 기대 조건을 선언한다. 이번 MiniPage의 `wait_for_selector`는 그 개념을 작게 만든 것이다. 학생은 “몇 초 기다렸는가”보다 “무엇이 보이면 성공인가”를 먼저 말할 수 있어야 한다. 이 습관이 없으면 자동화 코드가 실패할 때 timeout만 계속 늘리는 방향으로 흐른다.

## timeout 실패를 다루는 방법

TimeoutError를 숨기면 뒤 단계에서 더 큰 혼란이 생긴다. 없는 selector를 빈 문자열로 처리하면 나중에 CSV 저장이나 요약 단계에서 정상 데이터처럼 섞일 수 있다. 그래서 실패 케이스는 `ok: False` 또는 기대값이 빈 경우의 의도된 실패로 명확히 남긴다. `wait_cases.csv`에는 성공 케이스와 실패 케이스를 함께 넣어 학생이 양쪽 흐름을 모두 보게 한다.

실패를 기록할 때는 selector, timeout, expected_text, error_type을 남기면 좋다. 지금 문제에서는 단순한 ok 값으로 시작하지만, 빠른 학생에게는 실패 row에 `error` 필드를 추가하게 한다. 이 확장 과제는 실제 운영 자동화에서 장애 분석 시간을 줄이는 데 직접 도움이 된다.

## 동적 화면을 읽는 순서

동적 화면에서는 처음 DOM에 있는지와 실제로 보이는지가 다를 수 있다. 이 레슨의 `visible_elements`는 hidden 속성과 `data-delay-step`을 함께 본다. 학생이 `locator(...).elements()`로 전체 요소를 잡았다고 해서 모두 사용 가능한 상태는 아니다. 자동화에서는 “존재한다”, “보인다”, “클릭 가능하다”, “값이 바뀌었다”를 구분해야 한다.

대시보드에서 학생 row를 셀 때도 마찬가지다. row가 3개 있다는 사실과 현재 step에서 보이는 row 개수는 다를 수 있다. 수업 중에는 page.step을 일부러 출력하게 해 wait가 내부 상태를 어떻게 진행시키는지 확인한다. 이 작은 출력이 sleep과 wait의 차이를 이해시키는 데 효과적이다.

## selector 추천표를 남기는 이유

자동화 코드는 다음 사람이 유지보수한다. selector를 왜 골랐는지 기록하지 않으면 화면이 바뀌었을 때 어떤 기준으로 고쳐야 할지 알기 어렵다. 문제 13의 selector 추천표는 단순 표 만들기가 아니라 자동화 계약을 문서화하는 연습이다. purpose, selector, reason 세 칼럼으로 확장하면 더 좋다.

예를 들어 save 버튼은 `[data-testid="save-button"]`을 추천하고, lesson card는 `[data-testid="lesson-card"]`를 추천한다. 피해야 할 selector로 `.random-77` 같은 class를 기록하면 학생이 스스로 불안정한 기준을 구분할 수 있다. 이 습관은 실제 팀 프로젝트에서 QA와 프론트엔드 개발자가 소통할 때 중요하다.

## 실제 사이트로 확장할 때

실제 사이트에서는 동적 로딩이 더 복잡하다. 네트워크 요청이 늦거나, 권한에 따라 요소가 안 보이거나, 모바일/데스크톱 레이아웃이 달라질 수 있다. 그래서 selector와 wait 조건을 동시에 관리해야 한다. selector만 좋아도 조건이 틀리면 실패하고, wait만 길어도 selector가 틀리면 결국 실패한다.

수업에서는 외부 사이트를 반복 요청하지 않는다. fixture에서 selector와 wait 정책을 검증한 뒤, 실제 사이트에서는 테스트 계정과 낮은 요청 빈도로 확인한다. 개인정보가 있는 페이지는 로그에 원문을 남기지 않고, 필요한 경우 id나 상태값만 남긴다. 학생에게 자동화는 편리함보다 책임이 먼저라는 점을 함께 설명한다.

## selector 검토 예시

아래 기준은 학생 답안을 리뷰할 때 그대로 사용할 수 있다. 첫째, 저장 버튼은 `.btn.primary.x82`보다 `[data-testid="save-button"]`가 낫다. class 이름은 디자인 시스템이나 빌드 과정에서 바뀔 수 있지만 data-testid는 테스트 계약으로 남기는 값이기 때문이다. 둘째, 레슨 카드는 `.card`로도 잡히지만 카드가 아닌 다른 요소에 card class가 붙으면 결과가 섞일 수 있다. 그래서 `[data-testid="lesson-card"]`로 범위를 좁히고, 필요하면 `data-lesson-id`를 함께 읽는다.

셋째, 대시보드 metric은 텍스트 “제출 42” 자체를 selector로 삼지 않는다. 문구는 운영 상황에 따라 “제출 43”처럼 바뀐다. selector는 `[data-testid="metric-submit"]`로 잡고, 텍스트는 결과 검증에 사용한다. 넷째, 학생 row처럼 여러 개가 잡히는 selector는 하나만 읽는 코드와 전체를 읽는 코드를 구분한다. `first()`는 첫 요소만 확인할 때 쓰고, `elements()`나 `visible_elements()`는 목록을 검증할 때 쓴다.

## wait 케이스 설계법

wait_cases.csv는 단순히 정답을 담은 표가 아니다. 자동화 정책을 데이터로 표현한 문서다. selector, timeout_steps, expected_text를 분리해 두면 같은 코드를 여러 조건에 반복 적용할 수 있다. 성공 케이스만 넣으면 학생이 실패 처리의 필요성을 놓친다. 그래서 missing, ghost-panel처럼 의도적으로 실패해야 하는 케이스를 함께 넣는다. expected_text가 비어 있는 행은 “나타나지 않는 것이 정상”이라는 의미다.

이 구조는 실제 운영에서도 유용하다. 예를 들어 관리자 대시보드에서 오늘 수업 카드, 제출 통계, 오류 배너를 각각 기다려야 한다면 각 selector와 timeout 정책을 표로 관리할 수 있다. 테스트 코드가 길어질수록 조건을 코드 안에 흩뿌리는 것보다 표로 분리하는 편이 유지보수에 좋다.

## 학생에게 설명할 핵심 문장

“selector는 찾는 방법이고, wait는 찾을 때까지 기다리는 조건이다.” 이 한 문장을 반복해서 사용한다. 학생이 timeout을 늘리려고 할 때는 먼저 selector가 맞는지 묻는다. selector가 틀렸다면 아무리 기다려도 성공하지 않는다. 반대로 selector는 맞지만 아직 보이지 않는 상태라면 wait 조건이 필요하다. 두 문제를 구분하는 것이 이번 회차의 핵심이다.

또 하나의 핵심 문장은 “실패도 결과다”이다. 자동화에서 없는 요소를 찾지 못한 것은 코드가 망가졌다는 뜻일 수도 있지만, 의도된 실패 케이스를 검증한 것일 수도 있다. 중요한 것은 실패가 조용히 사라지지 않고 로그나 결과표에 남는 것이다. 학생 답안에서 try/except를 볼 때는 예외를 숨기는지, 결과 row에 남기는지 구분해서 피드백한다.

## 코드 리뷰 관점

좋은 7강 답안은 selector 선택 이유가 드러난다. data-testid를 썼는지, 결과 텍스트를 검증했는지, timeout 실패가 기록되는지, CSV 저장에 header가 있는지 본다. 특히 wait 문제는 현재 step을 출력하거나 결과 row에 남기면 이해도가 높다고 볼 수 있다. step을 보지 않고 최종 텍스트만 출력하면 기다림의 개념을 제대로 이해했는지 판단하기 어렵다.

보완이 필요한 답안은 보통 세 가지다. class selector만 사용한다. timeout을 크게 잡아 실패를 덮는다. CSV 저장 없이 print만 한다. 이 세 가지는 실제 운영 자동화에서 유지보수를 어렵게 만든다. 수업 중에는 틀렸다고만 말하지 말고, 왜 다음 주에 다시 실행하기 어려운지 연결해서 설명한다.


---

# 레슨 07 — 실습 문제 정답지


> 🔒 교사·관리자 전용. 학생에게 배포 금지.

안정적인 selector와 wait 실습 문제의 모범 답안이다. 출력값만 보지 말고 상태 변화, selector 안정성, 로그를 함께 확인한다.

## 0. 환경 셀


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/07/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

class MiniLocator:
    def __init__(self, page, selector):
        self.page = page
        self.selector = selector
    def elements(self):
        return self.page.soup.select(self.selector)
    def count(self):
        return len(self.elements())
    def first(self):
        items = self.elements()
        if not items:
            raise ValueError(f'no element for {self.selector}')
        return items[0]
    def text_content(self):
        return self.first().get_text(' ', strip=True)
    def all_text_contents(self):
        return [el.get_text(' ', strip=True) for el in self.elements()]
    def get_attribute(self, name):
        return self.first().get(name)
    def fill(self, value):
        self.first()['value'] = str(value)
    def click(self):
        return self.page._click(self.first())

class MiniPage:
    def __init__(self, html):
        self.soup = BeautifulSoup(html, 'html.parser')
        self.step = 0
        self.log = []
    def locator(self, selector):
        return MiniLocator(self, selector)
    def text_content(self, selector):
        return self.locator(selector).text_content()
    def fill(self, selector, value):
        self.locator(selector).fill(value)
        self.log.append({'action': 'fill', 'selector': selector, 'value': str(value)})
    def click(self, selector):
        result = self.locator(selector).click()
        self.log.append({'action': 'click', 'selector': selector, 'result': result})
        return result
    def _value(self, selector):
        el = self.soup.select_one(selector)
        return '' if el is None else el.get('value', '')
    def _click(self, el):
        action = el.get('data-action', '')
        if action == 'submit-profile':
            name = self._value('#student-name')
            course = self._value('#course-name')
            memo = self._value('#memo')
            out = self.soup.select_one('#result')
            out.string = f'{name} / {course} / {memo}'
            out['data-state'] = 'submitted'
            return 'submitted'
        if action == 'toggle-complete':
            target = self.soup.select_one(el.get('data-target', ''))
            if target:
                target['data-status'] = 'done' if target.get('data-status') != 'done' else 'pending'
                return target['data-status']
        if action == 'open-tab':
            target_id = el.get('data-target')
            for panel in self.soup.select('[role="tabpanel"]'):
                panel['hidden'] = 'true'
            target = self.soup.select_one(f'#{target_id}')
            if target and target.has_attr('hidden'):
                del target['hidden']
            return target_id
        return action or 'clicked'
    def visible_elements(self, selector):
        items = []
        for el in self.soup.select(selector):
            delay = int(el.get('data-delay-step', '0'))
            hidden = el.has_attr('hidden') or el.get('aria-hidden') == 'true'
            if delay <= self.step and not hidden:
                items.append(el)
        return items
    def tick(self):
        self.step += 1
        return self.step
    def wait_for_selector(self, selector, timeout_steps=5):
        for _ in range(timeout_steps + 1):
            items = self.visible_elements(selector)
            if items:
                return items[0]
            self.tick()
        raise TimeoutError(f'timeout waiting for {selector}')

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 문제 1 정답 — 안정 selector로 제목 읽기


In [ ]:
page = MiniPage(load_text('dynamic_dashboard.html'))
print(page.text_content('[data-testid="page-title"]'))


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. data-testid는 화면 문구나 임의 class보다 안정적인 기준이다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 2 정답 — 불안정 class와 안정 selector 비교


In [ ]:
lab = MiniPage(load_text('selector_lab.html'))
print(lab.locator('.card').count())
print(lab.locator('[data-testid="lesson-card"]').count())


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. class는 스타일 변경에 취약하고 data-testid는 테스트 목적이 분명하다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 3 정답 — 저장 버튼 selector 선택


In [ ]:
print(lab.locator('[data-testid="save-button"]').text_content())


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. 버튼 문구보다 data-testid가 유지보수에 적합하다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 4 정답 — 대시보드 즉시 보이는 metric


In [ ]:
print(page.step)
print(page.wait_for_selector('[data-testid="metric-active"]', timeout_steps=0).get_text(' ', strip=True))


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. delay 0 요소는 wait 없이도 바로 보인다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 5 정답 — 한 단계 뒤 나타나는 metric 기다리기


In [ ]:
el = page.wait_for_selector('[data-testid="metric-submit"]', timeout_steps=2)
print(page.step)
print(el.get_text(' ', strip=True))


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. wait는 조건이 충족될 때까지 확인 단계를 늘린다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 6 정답 — 두 단계 뒤 나타나는 metric 기다리기


In [ ]:
el = page.wait_for_selector('[data-testid="metric-pass"]', timeout_steps=3)
print(page.step)
print(el.get_text(' ', strip=True))


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. 느리게 표시되는 요소도 충분한 timeout이면 안정적으로 읽을 수 있다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 7 정답 — 학생 row 전체 대기 후 개수 세기


In [ ]:
page.step = 0
page.wait_for_selector('[data-testid="student-row"]', timeout_steps=2)
visible = page.visible_elements('[data-testid="student-row"]')
print(len(visible))


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. visible_elements는 현재 step에서 보이는 요소만 반환한다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 8 정답 — Timeout 오류 확인


In [ ]:
try:
    page.wait_for_selector('[data-testid="missing"]', timeout_steps=1)
except Exception as error:
    print(type(error).__name__)


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. 없는 selector는 명확히 실패해야 잘못된 자동화를 빨리 찾는다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 9 정답 — wait_cases CSV 읽기


In [ ]:
cases = list(csv.DictReader(load_text('wait_cases.csv').splitlines()))
print(len(cases))
print(cases[0]['selector'])


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. wait 검증도 데이터 케이스로 관리하면 반복 실행이 쉽다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 10 정답 — wait 케이스 실행하기


In [ ]:
results = []
for row in cases:
    p = MiniPage(load_text('dynamic_dashboard.html'))
    try:
        el = p.wait_for_selector(row['selector'], timeout_steps=int(row['timeout_steps']))
        text = el.get_text(' ', strip=True)
        results.append({'selector': row['selector'], 'ok': row['expected_text'] in text})
    except Exception:
        results.append({'selector': row['selector'], 'ok': row['expected_text'] == ''})
print(results)


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. 성공과 실패 케이스를 모두 기록해야 wait 정책을 검증할 수 있다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 11 정답 — 통과한 wait 케이스 수 세기


In [ ]:
passed = sum(row['ok'] for row in results)
print(passed, '/', len(results))


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. 테스트 결과는 개별 로그와 전체 통과 수를 함께 봐야 한다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 12 정답 — lesson-card id 목록 추출


In [ ]:
ids = [el['data-lesson-id'] for el in lab.locator('[data-testid="lesson-card"]').elements()]
print(ids)


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. 카드는 텍스트보다 data-lesson-id 같은 식별자를 저장하는 편이 안정적이다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 13 정답 — selector 추천표 만들기


In [ ]:
selector_rows = []
selector_rows.append({'purpose': 'save', 'selector': '[data-testid="save-button"]'})
selector_rows.append({'purpose': 'card', 'selector': '[data-testid="lesson-card"]'})
print(selector_rows)


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. 좋은 자동화는 selector 선택 이유를 기록한다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 14 정답 — selector 결과 CSV 저장


In [ ]:
with open('lesson07_wait_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['selector', 'ok'])
    writer.writeheader()
    writer.writerows(results)
print('saved:', len(results))


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. wait 결과를 CSV로 남겨야 어떤 selector가 흔들리는지 추적할 수 있다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

## 문제 15 정답 — 안정화 요약 문장 만들기


In [ ]:
print(f'wait 케이스 {passed}/{len(results)} 통과, 추천 selector {len(selector_rows)}개를 기록했습니다.')


### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 상태나 화면 구조를 먼저 명확한 자료구조로 바꾼 뒤 필요한 값만 선택한다. 마지막에는 숫자 근거가 들어간 운영 요약 문장으로 끝낸다. 결과가 맞아도 세션 상태, locator, wait, 로그가 코드에 남아 있지 않으면 운영형 자동화로 보기 어렵다.

### 채점 포인트

- 입력 fixture를 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 상태 변화, 버튼 클릭, wait 조건을 중간 변수나 로그로 확인할 수 있는가.
- selector가 화면 문구보다 id, data-testid, role 같은 안정적인 기준을 우선하는가.
- 출력 형태가 문제 요구사항과 일치하고 CSV 저장 시 헤더가 있는가.

### 자주 보이는 오답

- 화면 텍스트만 믿고 불안정한 selector를 사용한다.
- 상태가 바뀌기 전에 결과를 읽어 빈 값이나 이전 값이 나온다.
- 쿠키나 폼 입력 값을 문자열 변수에만 두고 세션/페이지 상태에 반영하지 않는다.
- 최종 미션에서 개별 문제 코드를 복사만 해서 함수화와 로그가 빠진다.

---

---

# 레슨 07 — 최종 미션 모범 답안


> 🔒 교사용. 학생에게는 최종 미션 문제 파일만 공유한다.

동적 대시보드 fixture에서 안정 selector와 wait 케이스를 검증하고 결과 CSV를 만든다.

## 0. 환경 셀


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/07/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

class MiniLocator:
    def __init__(self, page, selector):
        self.page = page
        self.selector = selector
    def elements(self):
        return self.page.soup.select(self.selector)
    def count(self):
        return len(self.elements())
    def first(self):
        items = self.elements()
        if not items:
            raise ValueError(f'no element for {self.selector}')
        return items[0]
    def text_content(self):
        return self.first().get_text(' ', strip=True)
    def all_text_contents(self):
        return [el.get_text(' ', strip=True) for el in self.elements()]
    def get_attribute(self, name):
        return self.first().get(name)
    def fill(self, value):
        self.first()['value'] = str(value)
    def click(self):
        return self.page._click(self.first())

class MiniPage:
    def __init__(self, html):
        self.soup = BeautifulSoup(html, 'html.parser')
        self.step = 0
        self.log = []
    def locator(self, selector):
        return MiniLocator(self, selector)
    def text_content(self, selector):
        return self.locator(selector).text_content()
    def fill(self, selector, value):
        self.locator(selector).fill(value)
        self.log.append({'action': 'fill', 'selector': selector, 'value': str(value)})
    def click(self, selector):
        result = self.locator(selector).click()
        self.log.append({'action': 'click', 'selector': selector, 'result': result})
        return result
    def _value(self, selector):
        el = self.soup.select_one(selector)
        return '' if el is None else el.get('value', '')
    def _click(self, el):
        action = el.get('data-action', '')
        if action == 'submit-profile':
            name = self._value('#student-name')
            course = self._value('#course-name')
            memo = self._value('#memo')
            out = self.soup.select_one('#result')
            out.string = f'{name} / {course} / {memo}'
            out['data-state'] = 'submitted'
            return 'submitted'
        if action == 'toggle-complete':
            target = self.soup.select_one(el.get('data-target', ''))
            if target:
                target['data-status'] = 'done' if target.get('data-status') != 'done' else 'pending'
                return target['data-status']
        if action == 'open-tab':
            target_id = el.get('data-target')
            for panel in self.soup.select('[role="tabpanel"]'):
                panel['hidden'] = 'true'
            target = self.soup.select_one(f'#{target_id}')
            if target and target.has_attr('hidden'):
                del target['hidden']
            return target_id
        return action or 'clicked'
    def visible_elements(self, selector):
        items = []
        for el in self.soup.select(selector):
            delay = int(el.get('data-delay-step', '0'))
            hidden = el.has_attr('hidden') or el.get('aria-hidden') == 'true'
            if delay <= self.step and not hidden:
                items.append(el)
        return items
    def tick(self):
        self.step += 1
        return self.step
    def wait_for_selector(self, selector, timeout_steps=5):
        for _ in range(timeout_steps + 1):
            items = self.visible_elements(selector)
            if items:
                return items[0]
            self.tick()
        raise TimeoutError(f'timeout waiting for {selector}')

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


## 모범 답안


In [ ]:
cases = list(csv.DictReader(load_text('wait_cases.csv').splitlines()))
results = []
for row in cases:
    p = MiniPage(load_text('dynamic_dashboard.html'))
    try:
        el = p.wait_for_selector(row['selector'], timeout_steps=int(row['timeout_steps']))
        text = el.get_text(' ', strip=True)
        results.append({'selector': row['selector'], 'ok': row['expected_text'] in text})
    except Exception:
        results.append({'selector': row['selector'], 'ok': row['expected_text'] == ''})
passed = sum(row['ok'] for row in results)
with open('lesson07_wait_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['selector','ok'])
    writer.writeheader(); writer.writerows(results)
print('passed:', passed, '/', len(results))


## 채점 메모

- 코드가 한 번 실행되어 산출물을 만들고, 다시 실행해도 같은 결과가 나와야 한다.
- 상태 변화, selector, 저장 경로, 수집 개수가 요약 문장과 충돌하지 않아야 한다.
- 실제 사이트로 옮길 때 필요한 요청 간격과 오류 처리 언급이 있어야 한다.

---

# 레슨 07 — 교사 가이드

## 학습 목표 (교사용)

- 불안정한 class selector와 안정적인 data-testid selector를 구분한다.
- 요소가 늦게 나타나는 상황을 wait로 처리한다.
- timeout 실패를 명확히 기록한다.
- wait 케이스를 CSV 기반으로 반복 검증한다.
- selector 추천표와 결과 로그를 저장한다.

## 2시간 수업 흐름

| 시간 | 운영 | 확인 포인트 |
|---:|---|---|
| 0-15분 | fixture 구조 읽기 | 상태·폼·selector 기준 확인 |
| 15-45분 | 강의 예제 실행 | 환경 셀과 helper 동작 확인 |
| 45-85분 | 문제 1~10 풀이 | 상태 변화와 반복 처리 확인 |
| 85-110분 | 문제 11~15 풀이 | 로그와 CSV 저장 확인 |
| 110-120분 | 최종 미션 정리 | 산출물과 요약 문장 검수 |

## 사전 준비

- 코랩 링크가 Kevin-innovation/jupyter-lecture 저장소를 가리키는지 확인한다.
- data 폴더의 fixture 파일을 먼저 열어 학생이 볼 태그와 상태 속성을 확인한다.
- 실제 사이트를 바로 요청하지 말고 합성 fixture로 구조를 읽게 한다.

## 질문 유도

- 상태가 코드 어디에 저장되는가?
- 화면 문구와 안정 selector 중 어느 쪽이 유지보수에 좋은가?
- 클릭 또는 입력 이후 어떤 값이 바뀌었는가?
- 이 자동화를 다음 주에도 실행한다면 어떤 로그가 필요한가?

## 채점 기준

15문제 중 12문제 이상 통과를 기본 완료로 본다. 최종 미션은 산출물 파일과 3문장 요약이 함께 있어야 완료 처리한다. 정답 코드와 다른 방식이어도 구조, 상태 변화, 출력 형태가 맞으면 인정한다.

## 자주 발생하는 오류

| 오류 | 원인 | 지도 방법 |
|---|---|---|
| 값이 비어 있음 | 입력 전 결과를 읽음 | fill/click 순서를 확인한다 |
| selector 오류 | 문구 기반 선택 | id, data-testid, role 기준으로 바꾼다 |
| wait 실패 | 아직 표시되지 않은 요소 읽음 | 조건 확인 후 wait를 사용한다 |
| 로그 누락 | 결과만 출력 | action/result를 리스트로 남기게 한다 |

## 확장 과제

결과를 CSV와 JSON 두 가지로 저장하거나, 처리 로그를 별도 리스트로 남기게 한다. 빠른 학생은 함수 분리와 오류 처리까지 진행한다.

## 운영 세부 가이드

7강은 “selector가 맞는가”와 “기다릴 조건이 맞는가”를 분리해서 지도해야 한다. 학생이 wait에 실패하면 timeout을 늘리기 전에 selector count를 먼저 확인하게 한다. selector가 0개면 기다려도 절대 성공하지 않는다. selector가 잡히지만 visible_elements에 나오지 않는다면 hidden이나 delay 조건을 확인한다.

## 문제별 채점 관찰 포인트

- 1~3번: data-testid와 class selector의 차이를 말로 설명할 수 있는가.
- 4~6번: delay-step에 맞춰 timeout을 설정했는가.
- 7번: locator 전체 개수와 현재 보이는 개수를 구분하는가.
- 8번: TimeoutError를 정상적인 실패 신호로 읽는가.
- 9~11번: CSV 케이스 전체를 반복하고 성공 수를 계산하는가.
- 12~13번: 카드 id와 selector 추천표를 안정 기준으로 작성하는가.
- 14~15번: CSV 저장과 요약 문장에 숫자 근거가 있는가.

## 수업 중 금지할 습관

- class 이름이 짧다는 이유만으로 핵심 selector로 쓰는 습관
- sleep 시간을 늘려서 실패를 덮는 습관
- TimeoutError를 빈 문자열로 바꿔 원인을 숨기는 습관
- 결과 파일 없이 화면 출력만 제출하는 습관
- 실제 사이트에서 요청 간격 없이 반복 실행하는 습관

## 빠른 학생 확장 과제

빠른 학생에게는 results row에 `timeout_steps`, `expected_text`, `actual_text`, `error_type`을 추가하게 한다. selector 추천표에는 `reason`과 `avoid` 칼럼을 추가하게 한다. 최종 미션에서는 CSV뿐 아니라 JSON 요약도 저장하게 하면 8강의 데이터 검증 흐름과 자연스럽게 이어진다.

## 느린 학생 보조 방식

느린 학생은 먼저 HTML에서 data-testid만 형광 표시하게 한다. 그 다음 wait_cases.csv의 selector를 하나씩 HTML에서 찾게 한다. 코드 작성 전에 “이 selector는 몇 step에 보이는가”를 표로 적게 하면 timeout 숫자를 찍는 문제가 아니라 조건을 이해하는 문제가 된다.

## 피드백 문장 예시

- selector는 안정적이지만 timeout 기준이 짧아 늦게 뜨는 요소를 놓치고 있다.
- 실패 케이스를 예외로 기록한 점은 좋다. 이제 어떤 selector가 실패했는지도 함께 저장해 보자.
- class selector는 현재는 동작하지만 배포 때 바뀔 수 있다. data-testid 기준으로 바꾸자.
- CSV 저장까지 했으니 요약 문장에 통과 수와 전체 수를 함께 넣으면 운영자가 읽기 쉽다.

## 판서 추천 구조

수업 중 칠판이나 공유 화면에는 다음 네 줄을 고정해 둔다.

1. selector count 확인
2. visible 상태 확인
3. wait 조건 확인
4. 결과 로그 저장

학생이 막힐 때마다 이 네 줄 중 어디에서 멈췄는지 표시하게 하면 질문이 구체화된다. “안 돼요”가 아니라 “selector count가 0이에요”, “count는 3인데 visible이 1개예요”, “timeout이 짧아요”처럼 말하게 만드는 것이 목표다.

## 제출물 검수 순서

먼저 학생용 노트북에 `____`가 남아 있는지 본다. 다음으로 results 길이가 wait_cases.csv 행 수와 같은지 본다. 그 다음 passed 값이 전체 케이스 기준으로 계산되었는지 확인한다. 마지막으로 CSV 파일을 열어 selector와 ok header가 있는지 확인한다. 이 순서를 지키면 결과 문자열 하나만 맞춘 답안을 걸러낼 수 있다.
